In [1]:
!pip -q install -U transformers datasets accelerate evaluate scikit-learn

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, Any, Optional

import torch
from torch.utils.data import Dataset

from sklearn.metrics import f1_score, precision_recall_fscore_support

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)

In [ ]:
# -------------------------
# Config
# -------------------------
MODEL_NAME = "xlm-roberta-base"
MAX_LEN = 256
SEED = 42
NUM_LABELS = 2
LR = 2e-5
EPOCHS = 4
TRAIN_BS = 16
EVAL_BS = 32
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
FP16 = True  # Kaggle T4/V100 supports fp16
GRAD_ACCUM = 1

set_seed(SEED)

In [2]:
# -------------------------
# Load data
# -------------------------
train_df = pd.read_csv("/kaggle/input/semeval-polarization/subtask1/train/ben.csv")
dev_df   = pd.read_csv("/kaggle/input/semeval-polarization/subtask1/dev/ben.csv")
test_df  = pd.read_csv("/kaggle/input/semeval-polarization/subtask1/test/ben.csv")

# Basic sanity
assert {"id", "text", "polarization"}.issubset(train_df.columns)
assert {"id", "text", "polarization"}.issubset(dev_df.columns)
assert {"id", "text"}.issubset(test_df.columns)

train_df["text"] = train_df["text"].astype(str)
dev_df["text"]   = dev_df["text"].astype(str)
test_df["text"]  = test_df["text"].astype(str)

# -------------------------
# Optional: quick label distribution print
# -------------------------
print("Train label distribution:\n", train_df["polarization"].value_counts(normalize=True))
print("Dev   label distribution:\n", dev_df["polarization"].value_counts(normalize=True))

In [8]:
# -------------------------
# Dataset wrapper
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

class TextClsDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int, has_labels: bool):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        text = row["text"]

        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding=False,  # handled by DataCollatorWithPadding
        )

        item = {k: torch.tensor(v) for k, v in enc.items()}

        if self.has_labels:
            item["labels"] = torch.tensor(int(row["polarization"]), dtype=torch.long)

        return item

train_ds = TextClsDataset(train_df, tokenizer, MAX_LEN, has_labels=True)
dev_ds   = TextClsDataset(dev_df, tokenizer, MAX_LEN, has_labels=True)
test_ds  = TextClsDataset(test_df, tokenizer, MAX_LEN, has_labels=False)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


,id,text,polarization
0,ben_1a632890da97e33a462ca67d4458a7f5,প্রিয় ভাই ও বোনেরা আমি ফুটবলের দেশ ব্রাজিলের ...,0
1,ben_a4957adc493a9488112c425df9882d8b,মাননীয় প্রধানমন্ত্রীর সুদৃষ্টি কামনা করছি কোটি...,0
2,ben_17467583659f68a038f13115d7fc3095,অতএব আমি তারকারাজির পতিত হবার স্থানের ব্ল্যাক ...,0
3,ben_a9827108a36f5f07eeff366f76d5451f,চোরকে বলে চুরি করতে গ্রিহস্তকে বলে সজাগ থাকতে ...,0
4,ben_adce136b077796d1ce9136812b6c2dbd,সত্যিই কি বি এন পি নেতৃত্ব সংকটে যারা বিএনপির ...,1


In [10]:
# -------------------------
# Metric + threshold tuning helpers
# -------------------------
def compute_metrics(eval_pred):
    """
    Returns Macro-F1 at default threshold 0.5 (argmax).
    For better results, we will separately tune threshold on dev after training.
    """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, preds, average="macro")
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {"macro_f1": macro_f1, "macro_precision": p, "macro_recall": r}

def find_best_threshold(probs_pos: np.ndarray, y_true: np.ndarray):
    """
    Sweep thresholds to maximize Macro-F1 on dev.
    probs_pos: probability of class 1
    """
    best_t, best_f1 = 0.5, -1.0
    # coarse-to-fine search
    for t in np.linspace(0.05, 0.95, 181):  # step 0.005
        y_pred = (probs_pos >= t).astype(int)
        f1 = f1_score(y_true, y_pred, average="macro")
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t), float(best_f1)

polarization
0    1909
1    1424
Name: count, dtype: int64
polarization
0    0.572757
1    0.427243
Name: proportion, dtype: float64


In [12]:
# -------------------------
# Model
# -------------------------
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
)


          char_len     word_len
count  3333.000000  3333.000000
mean    165.215722    29.147615
std      76.877219    12.942427
min      62.000000    18.000000
25%     115.000000    20.000000
50%     139.000000    25.000000
75%     187.000000    33.000000
max     636.000000    89.000000


In [13]:
# -------------------------
# Training args
# -------------------------
args = TrainingArguments(
    output_dir="./xlmr-ben-polarization",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    learning_rate=LR,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,

    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=EVAL_BS,
    gradient_accumulation_steps=GRAD_ACCUM,

    fp16=FP16,
    logging_steps=50,
    save_total_limit=2,
    report_to="none",  # disable wandb etc.
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# -------------------------
# Train
# -------------------------
trainer.train()

,count,mean,std,min,25%,50%,75%,max
polarization,,,,,,,,
0,1909.0,28.209534,11.991178,18.0,20.0,24.0,31.0,89.0
1,1424.0,30.405197,14.023652,18.0,21.0,25.0,35.0,89.0


In [14]:
# -------------------------
# Tune threshold on dev
# -------------------------
dev_out = trainer.predict(dev_ds)
dev_logits = dev_out.predictions
dev_labels = dev_out.label_ids

dev_probs = torch.softmax(torch.tensor(dev_logits), dim=-1).numpy()
dev_probs_pos = dev_probs[:, 1]

best_t, best_dev_f1 = find_best_threshold(dev_probs_pos, dev_labels)
print(f"Best dev threshold: {best_t:.3f} | Macro-F1: {best_dev_f1:.4f}")

# Also show argmax baseline on dev for comparison
dev_argmax = np.argmax(dev_logits, axis=-1)
print("Dev Macro-F1 (argmax):", f1_score(dev_labels, dev_argmax, average="macro"))

Random Macro-F1: 0.49829372544177564


In [ ]:
# -------------------------
# Predict test with tuned threshold
# -------------------------
test_out = trainer.predict(test_ds)
test_logits = test_out.predictions
test_probs = torch.softmax(torch.tensor(test_logits), dim=-1).numpy()
test_probs_pos = test_probs[:, 1]

test_pred = (test_probs_pos >= best_t).astype(int)

In [ ]:
# -------------------------
# Save submission
# -------------------------
sub = pd.DataFrame({
    "id": test_df["id"].values,
    "polarization": test_pred
})

sub_path = "submission_xlmr.csv"
sub.to_csv(sub_path, index=False)
print("Saved:", sub_path)
sub.head()